In [2]:
import json
from pathlib import Path

In [3]:
data = json.loads(Path("/mnt/sdd1/akanksha/formulacode/datasmith/src/datasmith/scrape/llm_inst/data.json").read_text())

In [4]:
data

{'00a45b4dca164105b50ba29e1735e96b573b639c': {'problem_statement': "The issue concerns an ambiguous documentation example for the `array_split` function. The task is to clarify the example to ensure it is easily understandable for users. There are no related issues or PRs mentioned. Acceptance criteria include a revised example that clearly demonstrates the function's usage without ambiguity. Assumptions include that the current example is confusing to users. Open questions include what specific aspects of the example are unclear. Next steps involve reviewing the current documentation, proposing a clearer example, and seeking feedback. The risk is minimal, primarily affecting user understanding.",
  'hints': 'The discussion involves a code review where the reviewer suggests removing trailing spaces and correcting indentations to align with the left of the ">>>". The reviewer also recommends reusing a variable for simplicity and advises using a more descriptive PR title in the future. T

In [3]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import pandas as pd

from datasmith import setup_environment
from datasmith.docker.context import ContextRegistry
from datasmith.notebooks.utils import merge_registries, update_cr

setup_environment()

/mnt/sdd1/atharvas/formulacode/datasmith
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
results_pth = Path("scratch/")

registries = results_pth.rglob("**/*context_registry*.json")
merged_json = merge_registries(list(registries))
cr = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))

scratch/context_registry_init.json : 5 entries
scratch/merged_context_registry_2025-09-04T08:32:08.486247.json : 140 entries
scratch/merged_context_registry_2025-09-09T14:32:37.382974.json : 4399 entries
scratch/merged_context_registry_2025-09-09T01:32:38.179134.json : 3253 entries
scratch/merged_context_registry_2025-09-05T20:02:28.617179.json : 540 entries
scratch/merged_context_registry_2025-09-06T01:31:46.096023.json : 700 entries
scratch/merged_context_registry_2025-09-06T21:27:11.754109.json : 1129 entries
scratch/merged_context_registry_2025-09-07T12:55:01.041054.json : 1184 entries
scratch/merged_context_registry_2025-09-04T23:54:53.035665.json : 178 entries
scratch/merged_context_registry_2025-09-06T06:14:21.165351.json : 975 entries
scratch/artifacts/pipeflush/symbolic_context_registry.json : 25 entries
scratch/artifacts/pipeflush/context_registry_filtered_perfonly.json : 551 entries
scratch/artifacts/pipeflush/context_registry.json : 7 entries
scratch/artifacts/processed/dow

In [5]:
commit_pth = Path("scratch/artifacts/pipeflush/commits_perfonly.parquet")
commit_df = pd.read_parquet(commit_pth)
commit_df.head()

,sha,date,message,total_additions,total_deletions,total_files_changed,files_changed,patch,has_asv,file_change_summary,kind,repo_name
0,3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8,2024-07-06T09:38:32+08:00,Merge pull request #125 from Kai-Striega/broad...,133,66,3,numpy_financial/_financial.py\nnumpy_financial...,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
1,3f67c275e1e575c902027ca07586b9d35f38033a,2024-05-07T15:04:23+10:00,Merge pull request #122 from Eugenia-Mazur/irr...,62,47,1,numpy_financial/_financial.py,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
2,5c66fb06ec95192d4b427b4de171b6ab9e1528a6,2024-05-04T11:03:28+10:00,Merge pull request #124 from Kai-Striega/confi...,8,18,3,asv.conf.json\ndoc/source/dev/running_the_benc...,From 646f292a26089dc212e4315f0939c183f660ccea ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
3,6c40b8efb727eacf8a865789afbe65ee2d4bb5c0,2024-04-04T14:13:19+11:00,Merge pull request #120 from Kai-Striega/enh/n...,6,2,1,numpy_financial/_cfinancial.pyx,From 5b134ac31419fea11db1dda25315d1bd192d8430 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
4,858358697fce8fb96530f9c299d285286e5192e5,2024-04-04T10:36:54+11:00,Merge pull request #118 from Kai-Striega/enh/n...,95,29,3,numpy_financial/_cfinancial.pyx\nnumpy_financi...,From 6b6f7b5ba1a50a1199c408b99538c397ef54d0ba ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial


In [6]:
import re
from typing import Final

# Words/phrases that very strongly indicate a performance change.
_POSITIVE_PATTERNS: Final = [
    r"\bperf(?:ormance)?\b",
    r"\boptimi[sz](?:e|ed|es|ation|ations|ing)?\b",
    r"\bspeed(?:\s*up|ed|ing)?\b",
    r"\bfaster\b",
    r"\blatency\b",
    r"\bthroughput\b",
    r"\boverhead\b",
    r"\bmemory(?:\s*(?:usage|footprint|alloc(?:ation|ations)?|pressure))?\b",
    r"\breduc(?:e|ed|es|ing)\s+(?:allocations?|copies|overhead|latency|cpu|memory)\b",
    r"\bavoid(?:ed|s|ing)?\s+(?:allocations?|copies)\b",
    r"\bcache(?:d|ing)?\b",
    r"\bvectori[sz]e(?:d|s|)?\b",
    r"\bparallel(?:ize|ism|ized|)\b",
    r"\bconcurren(?:t|cy)\b",
    r"\bprofil(?:e|ing|er)\b",
    r"\bbenchmark(?:s|ing)?\b",
    r"\bJIT\b|\bnumba\b|\bcython\b|\bsimd\b|\bavx\b|\bsse\b|\bneon\b",
    r"\bgpu\b|\bcuda\b|\bopencl\b",
    r"\bzero-?copy\b",
    r"\bpreallocat(?:e|ion)\b",
    r"\bhot(?:\s*path)?\b",
    r"\bbottleneck\b",
]

# Things that are almost never “performance optimizations” on their own.
_NEGATIVE_PATTERNS: Final = [
    # Meta / maintenance
    # r"^\s*merge (?:pull request|branch)\b",
    r"^\s*revert\b",
    r"^\s*(?:release|prepare(?:d)? release)\b|\bchangelog\b|\btag(?:ging)?\b",
    r"\bbump(?:ing)?\b|\bversions?\b",  # now matches "version" & "versions"
    r"\bupdate\s+versions?\b",  # e.g. "Update versions for 12.0.1"
    r"\[(?:\s*)release(?:\s*)\]",  # e.g. "[Release] ..."
    r"^\s*(?:minor|major|patch)\s*:?\s*\[?release\]?",  # e.g. "MINOR: [Release] ..."
    # Docs / typing / formatting / CI
    r"\bdocs?(?:umentation)?\b|\breadme\b|\bdocstring\b",
    r"\btype\s+comments?\b|\btype\s+annotations?\b|\btyping\b|\bmypy\b|\bpyright\b|\bpytype\b",
    r"\btypo\b",
    r"\bformat(?:ting)?\b|\bfmt\b|\blint(?:s|ing)?\b|\bblack\b|\bisort\b|\bruff\b|\bflake8\b",
    r"\bci\b|\bgithub actions\b|\bworkflow\b|\bpre-commit\b|\bcibuildwheel\b|\btravis\b|\bcircleci\b",
    r"\btests?\b|\bcoverage\b|\bTST:\b",
    # Infra / deps / packaging
    r"\bdependabot\b|\bdeps?\b|\bdependenc(?:y|ies)\b|\bpin(?:ning)?\b|\bunpin\b|\brequirements?\b|\bpyproject\.toml\b",
    r"\bbuild\b|\bwheels?\b|\bpackag(?:e|ing)\b|\bdocker\b|\bk8s\b|\bkubernetes\b|\bhelm\b",
    # Conventional-commits buckets that are rarely perf on their own
    r"^\s*chore\b",  # e.g. "chore(PageHeader): delete title param"
]

_POSITIVE_RE = re.compile("|".join(_POSITIVE_PATTERNS), re.I)
_NEGATIVE_RE = re.compile("|".join(_NEGATIVE_PATTERNS), re.I)


def basic_message_filter(msg: str) -> bool:
    """
    Returns True if the commit message looks performance-related (KEEP).
    Returns False if it's safe to filter out as not performance-related.

    Strategy:
      - If it hits any positive/perf signals -> keep.
      - Else if it hits any strong non-perf buckets -> filter out.
      - Else (ambiguous) -> keep (to avoid missing perf work).
    """
    if not msg:
        return False  # nothing useful -> filter

    if _POSITIVE_RE.search(msg):
        return True
    return not _NEGATIVE_RE.search(msg)

In [7]:
from datasmith.execution.filter_commits import crude_perf_filter

print(commit_df.shape)
filtered_df = crude_perf_filter(commit_df)
print(filtered_df.shape)

(46952, 12)
(12855, 15)


In [ ]:
# cr_path = commit_pth.parent / "context_registry_filtered_perfonly.json"
df_path = commit_pth.parent / "filtered_commits_perfonly.parquet"
print(f"Saving as {df_path}")
# print(f"Saving CR as {cr_path}")
filtered_df.to_parquet(df_path)
# new_cr.save_to_file(cr_path)

Saving as scratch/artifacts/pipeflush/filtered_commits_perfonly.parquet


: 

In [8]:
samples = filtered_df.loc[filtered_df.groupby("repo_name")["sha"].sample(frac=0.01).index][["repo_name", "sha"]]
samples.shape

(128, 2)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

from datasmith.execution.resolution import analyze_commit

all_results = []
with ThreadPoolExecutor(max_workers=128) as executor:
    futures = {
        executor.submit(analyze_commit, sha=row["sha"], repo_name=row["repo_name"]): row
        for _, row in samples.iterrows()
    }
    for future in as_completed(futures):
        row = futures[future]
        try:
            result = future.result()
            all_results.append(result)
            # print(f"Processed commit {row['sha']} from {row['repo_name']}: {result}")
        except Exception as e:
            print(f"Error processing commit {row['sha']} from {row['repo_name']}: {e}")

In [ ]:
# # len(all_results)
# for l in pd.DataFrame([r for r in all_results if r is not None]).dry_run_log:
#     if 'error' not in l: continue
#     print(l)

import ast

df = pd.DataFrame([r for r in all_results if r is not None])
df = df[df["can_install"]]


def repair_install_str(s: list[str]) -> list[str]:
    if isinstance(s, str) and s[0] == "[" and s[-1] == "]":
        s = ast.literal_eval(s)
    elif isinstance(s, str):
        s = [s]
    if not s:
        return []
    repaired = []
    for cmd in s:
        cmd = cmd.strip()
        cmd = cmd.removeprefix("in-dir={env_dir} ")
        cmd = cmd.replace("-mpip", "-m pip")
        repaired.append(cmd)
    return repaired


df["install_command"] = df["install_command"].apply(repair_install_str)
df

In [ ]:
from copy import deepcopy

filtered_shas = set(filtered_df["sha"])
new_cr = deepcopy(cr)
new_cr.registry = {t: c for t, c in cr.registry.items() if t.sha in filtered_shas}
len(new_cr.registry), len(cr.registry)

(551, 2879)

In [ ]:
from datasmith.execution.collect_commits_offline import find_parent_releases

parent_sha_dict = {}

for g, shas_series in filtered_df.groupby("repo_name")["sha"]:
    commits = list(set(shas_series))
    parents = find_parent_releases(repo_name=str(g), commits=commits, add_first=True, incl_datetime=True)
    parent_sha_dict[g] = dict(zip(commits, parents))

In [49]:
# parent_sha_dict.keys()
parent_sha_map = {s: p for d in parent_sha_dict.values() for s, (p, _) in d.items()}
parent_date_map = {s: d for d in parent_sha_dict.values() for s, (_, d) in d.items()}
filtered_df["base_sha"] = filtered_df["sha"].map(parent_sha_map)  # UNIX time

filtered_df["base_date"] = pd.to_datetime(filtered_df["sha"].map(parent_date_map), unit="s", origin="unix", utc=True)
# date is already in ISO format, convert to datetime
filtered_df["date"] = pd.to_datetime(filtered_df["date"], utc=True)

# make sure base_date and date are the same format
filtered_df

/tmp/ipykernel_117525/3648508101.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['base_sha'] = filtered_df['sha'].map(parent_sha_map) # UNIX time
/tmp/ipykernel_117525/3648508101.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['base_date'] = pd.to_datetime(filtered_df['sha'].map(parent_date_map), unit='s', origin='unix', utc=True)
/tmp/ipykernel_117525/3648508101.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[r

,sha,date,message,total_additions,total_deletions,total_files_changed,files_changed,patch,has_asv,file_change_summary,kind,repo_name,total_changes,n_files_changed,is_perf,parent_sha,base_sha,base_date
0,3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8,2024-07-06 01:38:32+00:00,Merge pull request #125 from Kai-Striega/broad...,133,66,3,numpy_financial/_financial.py\nnumpy_financial...,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial,199,3,True,8232cb495a4608d01396e9abbae953a2b294ba98,8232cb495a4608d01396e9abbae953a2b294ba98,2024-03-21 08:36:41+00:00
1,3f67c275e1e575c902027ca07586b9d35f38033a,2024-05-07 05:04:23+00:00,Merge pull request #122 from Eugenia-Mazur/irr...,62,47,1,numpy_financial/_financial.py,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial,109,1,True,553647c0e78e51cdb2abd03132585c4d8ddc208e,553647c0e78e51cdb2abd03132585c4d8ddc208e,2023-12-09 04:02:26+00:00
2,5c66fb06ec95192d4b427b4de171b6ab9e1528a6,2024-05-04 01:03:28+00:00,Merge pull request #124 from Kai-Striega/confi...,8,18,3,asv.conf.json\ndoc/source/dev/running_the_benc...,From 646f292a26089dc212e4315f0939c183f660ccea ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial,26,3,True,d60bf6d16844988c29c41df2a14fb5f17321ef29,d60bf6d16844988c29c41df2a14fb5f17321ef29,2024-04-03 06:00:43+00:00
3,6c40b8efb727eacf8a865789afbe65ee2d4bb5c0,2024-04-04 03:13:19+00:00,Merge pull request #120 from Kai-Striega/enh/n...,6,2,1,numpy_financial/_cfinancial.pyx,From 5b134ac31419fea11db1dda25315d1bd192d8430 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial,8,1,True,0b05d53f36a7a6831128004faed6d88293a35620,0b05d53f36a7a6831128004faed6d88293a35620,2024-01-09 22:21:12+00:00
4,858358697fce8fb96530f9c299d285286e5192e5,2024-04-03 23:36:54+00:00,Merge pull request #118 from Kai-Striega/enh/n...,95,29,3,numpy_financial/_cfinancial.pyx\nnumpy_financi...,From 6b6f7b5ba1a50a1199c408b99538c397ef54d0ba ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial,124,3,True,858358697fce8fb96530f9c299d285286e5192e5,858358697fce8fb96530f9c299d285286e5192e5,2024-04-03 23:36:54+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41581,e41a2fbda74a8b55f11b98d6262f1586f4566137,2014-06-12 23:37:51+00:00,releasing tardis 0.9.2\n,1,1,1,setup.py,From e41a2fbda74a8b55f11b98d6262f1586f4566137 ...,True,| File | Lines Added | Lines Removed | Total C...,tag,tardis-sn/tardis,2,1,True,35a0119dfc29d4bcceee859800c3bc77f0ec3d96,35a0119dfc29d4bcceee859800c3bc77f0ec3d96,2015-03-31 18:38:43+00:00
41587,eaab7a118c922fddf9e3844709cb67d898056b5c,2021-06-25 16:09:00+00:00,Use conda-forge packages instead of pypi (#166...,2,2,1,tardis_env3.yml,From eaab7a118c922fddf9e3844709cb67d898056b5c ...,True,| File | Lines Added | Lines Removed | Total C...,tag,tardis-sn/tardis,4,1,True,0349e7a90c90bfb9d9dbc5a106adfcbfda9a6f39,0349e7a90c90bfb9d9dbc5a106adfcbfda9a6f39,2024-08-02 19:48:15+00:00
41589,f47075deeb8b122a2224bcce40820c193ca03ad7,2020-06-27 20:11:55+00:00,Minor changes to the environment file and clea...,2,19,4,.readthedocs.yml\nazure-pipelines/simple_test_...,From f47075deeb8b122a2224bcce40820c193ca03ad7 ...,True,| File | Lines Added | Lines Removed | Total C...,tag,tardis-sn/tardis,21,4,True,643a6b1fa43540bcacbad0613ed1368e743632c9,643a6b1fa43540bcacbad0613ed1368e743632c9,2015-08-19 19:30:02+00:00
41591,f541e73b9c340c4250683fd6b567aa75984ee59b,2022-03-03 19:21:28+00:00,GPU Options (#1919)\n\n* Added parsing method ...,47,3,3,tardis/io/schemas/spectrum.yml\ntardis/monteca...,From f541e73b9c340c4250683fd6b567aa75984ee59b ...,True,| File | Lines Added | Lines Removed | Total C...,tag,tardis-sn/tardis,50,3,True,cf1178ac0af28712772b59cf6b098708ad2e9798,cf1178ac0af28712772b59cf6b098708ad2e9798,2019-03-10 15:03:27+00:00


In [148]:
cr_path = commit_pth.parent / "context_registry_filtered_perfonly.json"
df_path = commit_pth.parent / "filtered_commits_perfonly.parquet"
print(f"Saving as {df_path}")
print(f"Saving CR as {cr_path}")
filtered_df.to_parquet(df_path)
new_cr.save_to_file(cr_path)

Saving as scratch/artifacts/pipeflush/filtered_commits_perfonly.parquet
Saving CR as scratch/artifacts/pipeflush/context_registry_filtered_perfonly.json


00:23:45 INFO     datasmith.docker.context: Context registry saved to scratch/artifacts/pipeflush/context_registry_filtered_perfonly.json


In [ ]:
# # which of the filtered commits are already in context registry?
# # print(f"Already in context registry: {filtered_df[filtered_df['sha'].isin(cr.commits)].shape[0]} out of {filtered_df.shape[0]}")
# # [t.sha in filtered_df['sha'] for t in cr.registry]


(551, 2877)

In [ ]:
# # Break parquet file into 6 chunks. Try to put equal number of repos in each chunk.
# n_chunks = 6
# # (1036, 7)
# chunk_size = commit_df.shape[0] // n_chunks
# # shuffle rows and split into chunks
# commit_df = commit_df.sample(frac=1, random_state=42, replace=False).reset_index(drop=True)
# chunks = [commit_df.iloc[i * chunk_size : (i + 1) * chunk_size] for i in range(n_chunks - 1)]
# chunks.append(commit_df.iloc[(n_chunks - 1) * chunk_size :])  # last chunk gets the remainder
# cmds = []
# for i, chunk in enumerate(chunks):
#     pth = Path(f"scratch/artifacts/pipeflush/chunk_{i}/commits_perfonly.parquet")
#     pth.parent.mkdir(parents=True, exist_ok=True)
#     chunk.to_parquet(pth)
#     # Make a new context registry:
#     cr.save_to_file(pth.parent / "context_registry.json")
#     cmd_i = cmd.format(output_dir=pth.parent)
#     cmds.append(cmd_i)

In [ ]:
# # break parquet into three chunks with fixed ratios.
# ratios = [64, 56, 127]
# total = sum(ratios)
# # Compute split sizes
# sizes = [int(commit_df.shape[0] * r / total) for r in ratios]

# # Adjust last size to cover remainder (to avoid row loss due to rounding)
# sizes[-1] = commit_df.shape[0] - sum(sizes[:-1])

# # Split dataframe
# df1 = commit_df.iloc[: sizes[0]]
# df2 = commit_df.iloc[sizes[0] : sizes[0] + sizes[1]]
# df3 = commit_df.iloc[sizes[0] + sizes[1] :]

# chunks = [df1, df2, df3]
# cmds = []
# for i, (_, ratio) in enumerate(zip(chunks, ratios)):
#     pth = Path(f"scratch/artifacts/pipeflush/chunk_{i}/commits_perfonly.parquet")
#     pth.parent.mkdir(parents=True, exist_ok=True)
#     # chunk.to_parquet(pth)
#     # Make a new context registry:
#     # cr.save_to_file(pth.parent / "context_registry.json")
#     cmd_i = cmd.format(output_dir=pth.parent, ncpus=(ratio // 2))
#     cmds.append(cmd_i)

In [ ]:
# cmds is defined in the commented code above
# Uncomment the code in cells 11-12 to define cmds before running this
# print("\n".join(cmds).replace("  ", ""))

python scratch/scripts/synthesize_contexts.py --commits scratch/artifacts/pipeflush/chunk_0/commits_perfonly.parquet --output-dir scratch/artifacts/pipeflush/chunk_0/results_synthesis/ --context-registry scratch/artifacts/pipeflush/chunk_0/context_registry.json --max-workers 32 --limit-per-repo 2 --max-attempts 3 --max-steps 10
python scratch/scripts/synthesize_contexts.py --commits scratch/artifacts/pipeflush/chunk_1/commits_perfonly.parquet --output-dir scratch/artifacts/pipeflush/chunk_1/results_synthesis/ --context-registry scratch/artifacts/pipeflush/chunk_1/context_registry.json --max-workers 28 --limit-per-repo 2 --max-attempts 3 --max-steps 10
python scratch/scripts/synthesize_contexts.py --commits scratch/artifacts/pipeflush/chunk_2/commits_perfonly.parquet --output-dir scratch/artifacts/pipeflush/chunk_2/results_synthesis/ --context-registry scratch/artifacts/pipeflush/chunk_2/context_registry.json --max-workers 63 --limit-per-repo 2 --max-attempts 3 --max-steps 10


In [ ]:
# # make a tiny task with just two commits for testing
# tiny = commit_df.sample(n=2, random_state=941, replace=False).reset_index(drop=True)
# tiny_pth = Path("scratch/artifacts/pipeflush/tiny/commits_perfonly.parquet")
# tiny_pth.parent.mkdir(parents=True, exist_ok=True)
# tiny.to_parquet(tiny_pth)
# # cr.save_to_file(tiny_pth.parent / "context_registry.json")
# cmd_tiny = cmd.format(output_dir=tiny_pth.parent, ncpus=2)
# print(cmd_tiny)

python scratch/scripts/synthesize_contexts.py     --commits scratch/artifacts/pipeflush/tiny/commits_perfonly.parquet     --output-dir scratch/artifacts/pipeflush/tiny/results_synthesis/     --context-registry scratch/artifacts/pipeflush/tiny/context_registry.json     --max-workers 2     --limit-per-repo 2     --max-attempts 3     --max-steps 10


In [ ]:
# import numpy as np

# lens = np.array([len(d) for d in [df1, df2, df3]])
# ratios = np.array(ratios)

# l2r = lens * 15 / ratios  # min
# l2r_hrs = l2r / 60
# l2r_hrs

array([47.51953125, 47.51785714, 47.52559055])